# Quickstart: 16 cuts as a sum of small trees

A single decision tree with a small budget spends its cuts copying the same effect
into different branches. `InterleavedTreeClassifier` spends the **same number of cuts** on a
logit sum of a few small trees, and the result still reads as a scorecard.

The data are simulated credit defaults (`credit_data.py`): the true model is a sum of
one-feature effects plus one interaction, so we know what a good model should find.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

from credit_data import make_credit_data

X, y = make_credit_data(n=20_000, seed=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X_train):,} training rows, {len(X_test):,} test rows, "
      f"{y.mean():.1%} defaults, {X['income'].isna().mean():.0%} incomes missing")

14,000 training rows, 6,000 test rows, 10.3% defaults, 5% incomes missing


## Same budget, two shapes

Both models get 16 cuts.

In [2]:
from sklearn.tree import DecisionTreeClassifier

from bettertrees import BudgetClassifier, InterleavedTreeClassifier, SumOfOptimalTrees

models = {
    "CART, 16 cuts (17 leaves)": DecisionTreeClassifier(
        max_leaf_nodes=17, min_samples_leaf=50, random_state=0),
    "BudgetClassifier, 16 cuts (the no-tuning rule)": BudgetClassifier(max_splits=16),
    "ITM, 16 cuts": InterleavedTreeClassifier(max_splits=16),
    "sum of optimal depth-2 trees, 16 cuts": SumOfOptimalTrees(
        n_trees=5, depth=2, extra_stumps=1),
}
rows = []
for name, model in models.items():
    p = model.fit(X_train, y_train).predict_proba(X_test)[:, 1]
    rows.append(dict(model=name, auc=roc_auc_score(y_test, p), log_loss=log_loss(y_test, p)))
pd.DataFrame(rows).set_index("model").round(4)

,auc,log_loss
model,,
"CART, 16 cuts (17 leaves)",0.6688,0.3142
"BudgetClassifier, 16 cuts (the no-tuning rule)",0.6956,0.3085
"ITM, 16 cuts",0.7029,0.3072
"sum of optimal depth-2 trees, 16 cuts",0.6843,0.3111


## Reading the model

`explain()` prints the whole model. To score a row, start from the base, add one
value per tree (the leaf the row falls into) and apply the sigmoid.

In [3]:
itm = models["ITM, 16 cuts"]
print(itm.explain())

logit P(y = 1) = base -2.1659 + sum of 8 trees (16 cuts)
tree 1:
  -0.3639  if late_payments <= 0.5
  +0.3077  if late_payments > 0.5 and debt_ratio <= 0.4325
  +0.9798  if late_payments > 0.5 and debt_ratio > 0.4325
tree 2:
  +0.4033  if (income <= 3.635e+04 or missing) and credit_lines <= 4.5
  +0.0971  if (income <= 3.635e+04 or missing) and credit_lines > 4.5
  -0.0432  if 3.635e+04 < income <= 5.085e+04
  -0.5047  if income > 5.085e+04
tree 3:
  -0.1066  if late_payments <= 1.5
  +0.7493  if late_payments > 1.5
tree 4:
  +0.4383  if months_employed <= 12.5 and credit_lines <= 6.5
  -0.0227  if months_employed <= 12.5 and credit_lines > 6.5
  +0.2795  if months_employed > 12.5 and age <= 25.5
  -0.1646  if months_employed > 12.5 and age > 25.5
tree 5:
  -0.1296  if age <= 59.5
  +0.1048  if 59.5 < age <= 73.5
  +0.4337  if age > 73.5
tree 6:
  +0.3009  if (income <= 1.715e+04 or missing)
  -0.0526  if income > 1.715e+04
tree 7:
  +0.0945  if debt_ratio <= 0.5205 and credit_lines <=

Tree 1 found the interaction we planted: a high debt ratio matters much more for people
who already paid late. The other trees are mostly single effects (income, employment,
age), which a single tree would have had to repeat in several branches.

In [4]:
row = X_test.iloc[[0]]
contrib = itm.predict_contributions(row)[0]
logit = itm.base_margin_ + contrib.sum()
print(row.T.rename(columns={row.index[0]: "value"}))
print(f"\nbase {itm.base_margin_:+.3f} + trees {' '.join(f'{c:+.3f}' for c in contrib)}")
print(f"= logit {logit:+.3f} -> P(default) = {1 / (1 + np.exp(-logit)):.1%}")
print(f"predict_proba: {itm.predict_proba(row)[0, 1]:.1%}")

                  value
age              78.000
income              NaN
debt_ratio        0.411
late_payments     1.000
credit_lines      5.000
months_employed   2.000
has_mortgage      0.000

base -2.166 + trees +0.308 +0.097 -0.107 +0.438 +0.434 +0.301 -0.120 +0.095
= logit -0.720 -> P(default) = 32.7%
predict_proba: 32.7%


## Choosing the budget

The budget is the one knob. It works with the usual scikit-learn tools.

In [5]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(InterleavedTreeClassifier(), {"max_splits": [4, 8, 16, 32]},
                      scoring="neg_log_loss", cv=3).fit(X_train, y_train)
pd.DataFrame(search.cv_results_)[["param_max_splits", "mean_test_score"]]

,param_max_splits,mean_test_score
0,4,-0.313110
1,8,-0.309036
2,16,-0.309443
3,32,-0.311665
